# Modelado de Área de Incendios con KNN
Este cuaderno demuestra cómo aplicar **K-Nearest Neighbors (KNN)** para modelar la extensión de incendios forestales.

## ¿Cómo funciona KNN?
- KNN es un algoritmo de aprendizaje supervisado que, para predecir el valor de una instancia, busca los *k* vecinos más cercanos en el espacio de características.
- La cercanía se mide mediante una métrica de distancia (euclídea, Manhattan, etc.).
- En regresión, la predicción es la media (o ponderación) de los valores de los vecinos.
- Es **no paramétrico**, sin supuestos fuertes sobre la distribución de los datos.


In [48]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedKFold
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error


## 1. Carga de Datos y Transformación del Target
Cargamos el dataset y se aplicamos **transformación logarítmica** a la variable objetivo (`area`) para reducir asimetrías y valores extremos.


In [49]:
data = pd.read_csv('forestfires.csv')  # Asume forestfires.csv en el directorio
data['area_log'] = np.log1p(data['area'])  # Transformación: log(1 + area)
y = data['area_log']  # Target log-transformado
X = data.drop(['area', 'area_log'], axis=1)  # Características
display(X.head())
display(y.head())


,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0


0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: area_log, dtype: float64

## 2. Definición de Categorías para OneHotEncoder
Definimmos explícitamente las categorías para `month` y `day` para la realizar la codificacion OneHot.


In [50]:
month_categories = ['jan','feb','mar','apr','may','jun',
                    'jul','aug','sep','oct','nov','dec']
day_categories = ['mon','tue','wed','thu','fri','sat','sun']
print('Meses:', month_categories)
print('Días:', day_categories)

Meses: ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
Días: ['mon', 'tue', 'wed', 'thu', 'fri', 'sat', 'sun']


## 3. Preprocesamiento
- **RobustScaler**: Escala variables numéricas reduciendo el efecto de outliers.
- **OneHotEncoder**: Codifica variables categóricas con manejo de categorías desconocidas.


In [51]:
'''preprocessor = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), ['FFMC','DMC','DC','ISI','temp','RH','wind','rain']),
        ('cat', OneHotEncoder(categories=[month_categories, day_categories], handle_unknown='ignore'), ['month','day'])
    ]
)
preprocessor

print(data)'''

X = pd.get_dummies(X, columns=['month', 'day'], drop_first = True).astype(float)

print(X)



       X    Y  FFMC    DMC     DC   ISI  temp    RH  wind  rain  ...  \
0    7.0  5.0  86.2   26.2   94.3   5.1   8.2  51.0   6.7   0.0  ...   
1    7.0  4.0  90.6   35.4  669.1   6.7  18.0  33.0   0.9   0.0  ...   
2    7.0  4.0  90.6   43.7  686.9   6.7  14.6  33.0   1.3   0.0  ...   
3    8.0  6.0  91.7   33.3   77.5   9.0   8.3  97.0   4.0   0.2  ...   
4    8.0  6.0  89.3   51.3  102.2   9.6  11.4  99.0   1.8   0.0  ...   
..   ...  ...   ...    ...    ...   ...   ...   ...   ...   ...  ...   
512  4.0  3.0  81.6   56.7  665.6   1.9  27.8  32.0   2.7   0.0  ...   
513  2.0  4.0  81.6   56.7  665.6   1.9  21.9  71.0   5.8   0.0  ...   
514  7.0  4.0  81.6   56.7  665.6   1.9  21.2  70.0   6.7   0.0  ...   
515  1.0  4.0  94.4  146.0  614.7  11.3  25.6  42.0   4.0   0.0  ...   
516  6.0  3.0  79.5    3.0  106.7   1.1  11.8  31.0   4.5   0.0  ...   

     month_may  month_nov  month_oct  month_sep  day_mon  day_sat  day_sun  \
0          0.0        0.0        0.0        0.0      0.0 

## 4. Pipeline y Validación Cruzada
- Creamos un **Pipeline** que aplicando el preprocesamiento y luego KNN de regersión.
- Se utiliza **RepeatedKFold** con 10 folds y 30 repeticiones para tener una validación robusta.


In [52]:
'''model = Pipeline([
    ('preprocessor', X),
    ('regressor', KNeighborsRegressor())
])'''
model = Pipeline([
    ("scaler", RobustScaler()),
    ("regressor", KNeighborsRegressor())
])

cv = RepeatedKFold(n_splits=5, n_repeats=30, random_state=42)
model, cv

(Pipeline(steps=[('scaler', RobustScaler()),
                 ('regressor', KNeighborsRegressor())]),
 RepeatedKFold(n_repeats=30, n_splits=5, random_state=42))

## 5. Búsqueda de Hiperparámetros
- Probamos con distintos valores de `n_neighbors(k)`, `weights` y `metric`.
- **GridSearchCV** con métrica `neg_mean_squared_error` porque asume que el MSE mas alto es el
     mejor, asi que usamos el negativo.


In [58]:
param_grid = {
    'regressor__n_neighbors': range(100, 300),
    'regressor__weights': ['uniform', 'distance'],
    'regressor__metric': ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(model, param_grid, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search.fit(X, y)
print('Mejores parámetros:', grid_search.best_params_)

Mejores parámetros: {'regressor__metric': 'manhattan', 'regressor__n_neighbors': 177, 'regressor__weights': 'distance'}


## 6. Evaluación Final
- Se evaluúa el mejor modelo sobre todo el dataset (predicciones y anti-transformación).
- Cálculo de **MSE**, **MAE** y **R²** en la escala original:
- **MSE** (Mean Squared Error): promedio de los cuadrados de los errores (hectareas²).
- **MAE** (Mean Absolute Error): promedio de los valores absolutos de los errores en hectareas.
- **R²**  (coeficiente de determinación): proporción de la varianza explicada por el modelo (adimensional, de 0 a 1).
- **Error absoluto** diferencia |predicción – valor real|, medido en hectáreas.
- **% de error** (error absoluto / valor real) × 100, un porcentaje que indica el tamaño del error relativo al valor real.


In [59]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X)
y_pred_exp = np.expm1(y_pred)
y_true_exp = np.expm1(y)

mse = mean_squared_error(y_true_exp, y_pred_exp)
mae = mean_absolute_error(y_true_exp, y_pred_exp)
r2 = r2_score(y_true_exp, y_pred_exp)

print(f'MSE: {mse:.2f}')
print(f'MAE: {mae:.2f}')
print(f'R²: {r2:.2f}')

MSE: 0.47
MAE: 0.10
R²: 1.00


## 7. Predicciones de Ejemplo
Cogemos 5 predicciones aleatorias del dataset y cálculamos la prediccion del aera quemada, el error absoluto y el error relativo de cada una.


In [60]:
random_indices = random.sample(range(len(X)), 5)
for i, idx in enumerate(random_indices, 1):
    sample = X.iloc[idx:idx+1]
    real_val = y_true_exp[idx]
    pred = np.expm1(best_model.predict(sample))[0]
    err = abs(pred - real_val)
    print(f'Muestra {i} (índice {idx}): ValorReal={real_val:.2f} hectareas, Predicción={pred:.2f} hectareas, Error abs={err:.2f}, % Error={(err/(real_val+1e-6))*100:.1f}%')

Muestra 1 (índice 16): ValorReal=0.00 hectareas, Predicción=0.00 hectareas, Error abs=0.00, % Error=0.0%
Muestra 2 (índice 447): ValorReal=0.00 hectareas, Predicción=0.00 hectareas, Error abs=0.00, % Error=0.0%
Muestra 3 (índice 445): ValorReal=0.00 hectareas, Predicción=0.00 hectareas, Error abs=0.00, % Error=0.0%
Muestra 4 (índice 105): ValorReal=0.00 hectareas, Predicción=0.00 hectareas, Error abs=0.00, % Error=0.0%
Muestra 5 (índice 439): ValorReal=0.33 hectareas, Predicción=0.33 hectareas, Error abs=0.00, % Error=0.0%
